# Qwen 2.5 72B on 2 GPUs

Notebook optimisé pour `Qwen/Qwen2.5-72B-Instruct` sur **2 GPUs**.

Objectifs:
- variables de configuration séparées et faciles à modifier
- vérification GPU / mémoire après chargement du modèle
- smoke test sur 20 lignes
- mesure du débit (`rows/s`) et du parsing
- indicateur simple de convergence / stabilité
- run complet exportable
- sanity check final
- graphiques de comparaison multi-modèles si plusieurs CSV existent déjà


## 1) Imports

In [ ]:
import os
import gc
import json
import math
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from vllm import LLM, SamplingParams


## 2) Runtime and GPU configuration

In [ ]:
# GPU visibility: set exactly the two GPUs you want to use.
CUDA_VISIBLE_DEVICES = "0,1"
TENSOR_PARALLEL_SIZE = 2

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

conda_prefix = os.environ.get("CONDA_PREFIX", "")
if conda_prefix:
    conda_lib = f"{conda_prefix}/lib"
    ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    ld_parts = [p for p in ld_library_path.split(":") if p]
    if conda_lib not in ld_parts:
        os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_library_path}" if ld_library_path else conda_lib

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TENSOR_PARALLEL_SIZE:", TENSOR_PARALLEL_SIZE)


## 3) Paths

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv").exists():
    PROJECT_ROOT = Path("/Users/raresolteanu/Desktop/Gliner-Work.Dauphine")

DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "communication_function_outputs" / "qwen72b_2gpu"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LATEX_DIR = PROJECT_ROOT / "latex"
LATEX_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## 4) Model configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
CHAT_MODE = "hf_auto"
MODEL_SLUG = "qwen_qwen2_5_72b_instruct"

# Generation
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 220

# Memory / context
VLLM_GPU_MEMORY_UTILIZATION = 0.90
MAX_MODEL_LEN = 8192
VLLM_SWAP_SPACE_GB = 16
MODEL_DTYPE = "bfloat16"
DISABLE_CUSTOM_ALL_REDUCE = False

# Execution
SMOKE_TEST_N = 20
FULL_RUN_N = None   # None = all available rows
PROMPT_BATCH_LABEL = "qwen72b_2gpu_v1"

MODEL_KWARGS: dict[str, Any] = {
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "trust_remote_code": True,
    "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
    "dtype": MODEL_DTYPE,
    "max_model_len": int(MAX_MODEL_LEN),
    "swap_space": int(VLLM_SWAP_SPACE_GB),
}
if DISABLE_CUSTOM_ALL_REDUCE:
    MODEL_KWARGS["disable_custom_all_reduce"] = True

print("MODEL_NAME:", MODEL_NAME)
print("CHAT_MODE:", CHAT_MODE)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("MAX_MODEL_LEN:", MAX_MODEL_LEN)
print("VLLM_GPU_MEMORY_UTILIZATION:", VLLM_GPU_MEMORY_UTILIZATION)


## 5) Input strategy

In [ ]:
# Choose one:
# - "visual_only"
# - "all_french_columns"
# - "long_french_text_only"

COLUMN_STRATEGY = "all_french_columns"
MIN_TOKENS_PER_FIELD = 7
TEXT_COLUMNS_FR = ["Script", "Incrustation", "Titre", "Visuel"]

def clean_text(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_count(text: str) -> int:
    return 0 if not text else len(text.split())

def join_labeled_parts(parts: list[tuple[str, str]]) -> str:
    kept = []
    for label, text in parts:
        text = clean_text(text)
        if text:
            kept.append(f"{label}: {text}")
    return "\n".join(kept).strip()

def build_input_text(row: pd.Series) -> str:
    fr_parts = [(col, row.get(col, "")) for col in TEXT_COLUMNS_FR]
    if COLUMN_STRATEGY == "visual_only":
        return clean_text(row.get("Visuel", ""))
    if COLUMN_STRATEGY == "all_french_columns":
        return join_labeled_parts(fr_parts)
    if COLUMN_STRATEGY == "long_french_text_only":
        selected = []
        for col in TEXT_COLUMNS_FR:
            text = clean_text(row.get(col, ""))
            if token_count(text) > MIN_TOKENS_PER_FIELD:
                selected.append((col, text))
        return join_labeled_parts(selected)
    raise ValueError(f"Unknown COLUMN_STRATEGY: {COLUMN_STRATEGY}")


## 6) Prompt and parser

In [ ]:
RUBRIC_TEXT = '''You score French automotive ads on 3 dimensions from 1 to 5.

- informativeness: factual, technical, concrete product information, offer details, equipment, performance specs, financing or practical product information.
- expressiveness: emotion, desire, identity, style, seduction, aspiration, strong aesthetic or affective framing.
- phatic: social bond, relational tone, conversational closeness, bonding language, direct relationship-maintaining communication.

Rules:
- Score each dimension independently from 1 to 5.
- Use mixed for dominant_dimension only if the highest score is tied.
- dominant_dimension_score must equal the highest score.
- Keep the reason very short and concrete.
- Return strict JSON only.
'''

DIMENSIONS = ["informativeness", "expressiveness", "phatic"]
SCORE_MIN = 1
SCORE_MAX = 5

def output_schema_text() -> str:
    return '''{
  "informativeness": 1,
  "expressiveness": 1,
  "phatic": 1,
  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",
  "dominant_dimension_score": 1,
  "confidence": 0.0,
  "reason": "short explanation"
}'''

def build_prompt_content(ad_text: str) -> str:
    clean_ad_text = " ".join(str(ad_text).split()).strip()
    return (
        f"{RUBRIC_TEXT}\n\n"
        f"Return strict JSON only with:\n{output_schema_text()}\n\n"
        f"Ad to score:\n{clean_ad_text}\n\n"
        "Return only strict JSON."
    )

def render_prompt_for_model(content: str, llm=None) -> str:
    if CHAT_MODE == "plain" or llm is None:
        return content
    tokenizer = llm.get_tokenizer()
    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_prompt(ad_text: str, llm=None) -> str:
    return render_prompt_for_model(build_prompt_content(ad_text), llm=llm)

def clamp_score(value, default=1):
    try:
        score = int(round(float(value)))
    except Exception:
        score = default
    return max(SCORE_MIN, min(SCORE_MAX, score))

def normalize_dimension_name(value):
    text = str(value or "").strip().lower()
    aliases = {
        "informative": "informativeness",
        "information": "informativeness",
        "referential": "informativeness",
        "expressive": "expressiveness",
        "emotive": "expressiveness",
        "emotion": "expressiveness",
        "phatique": "phatic",
    }
    text = aliases.get(text, text)
    return text if text in DIMENSIONS or text == "mixed" else ""

def extract_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip())
                if isinstance(payload, dict):
                    return payload
            except json.JSONDecodeError:
                continue
    raise ValueError("Could not extract JSON from model output.")

def parse_model_prediction(raw_text: str):
    try:
        payload = extract_json_object(raw_text)
        scores = {
            "informativeness": clamp_score(payload.get("informativeness", 1)),
            "expressiveness": clamp_score(payload.get("expressiveness", 1)),
            "phatic": clamp_score(payload.get("phatic", 1)),
        }
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_score = ranked[0][1]
        top_labels = [k for k, v in ranked if v == top_score]
        dominant_dimension = normalize_dimension_name(payload.get("dominant_dimension")) or ("mixed" if len(top_labels) > 1 else top_labels[0])
        reason = " ".join(str(payload.get("reason", "")).split()).strip() or "strict_json"
        try:
            confidence = round(float(payload.get("confidence", 0.5)), 4)
        except Exception:
            confidence = 0.5
        confidence = max(0.0, min(1.0, confidence))
        return {
            "informativeness": scores["informativeness"],
            "expressiveness": scores["expressiveness"],
            "phatic": scores["phatic"],
            "dominant_dimension": dominant_dimension,
            "dominant_dimension_score": top_score,
            "confidence": confidence,
            "reason": reason,
        }, True, ""
    except Exception as exc:
        return {
            "informativeness": 1,
            "expressiveness": 1,
            "phatic": 1,
            "dominant_dimension": "mixed",
            "dominant_dimension_score": 1,
            "confidence": 0.0,
            "reason": "parse_fallback",
        }, False, str(exc)


## 7) Load data

In [ ]:
df = pd.read_csv(DATASET_PATH)
required_cols = ["row_id"] + [c for c in TEXT_COLUMNS_FR if c in df.columns]
sample_df = df.loc[:, required_cols].copy()
sample_df["model_input_text"] = sample_df.apply(build_input_text, axis=1)
sample_df = sample_df[sample_df["model_input_text"].str.strip() != ""].copy()

if FULL_RUN_N is not None:
    sample_df = sample_df.head(FULL_RUN_N).copy()

print("Rows available:", len(sample_df))
display(sample_df[["row_id", "model_input_text"]].head(3))


## 8) Sampling params and helper boxes

In [ ]:
sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_NEW_TOKENS,
)

def html_box(title: str, body: str, color: str = "#1f4e79", bg: str = "#eef6fb"):
    display(HTML(
        f"""
        <div style="border-left: 6px solid {color}; background:{bg}; padding:10px 14px; margin:8px 0; border-radius:6px;">
            <div style="font-weight:700; margin-bottom:4px;">{title}</div>
            <div style="white-space:pre-wrap;">{body}</div>
        </div>
        """
    ))


## 9) GPU inspection after model load

In [ ]:
def inspect_gpu_state():
    try:
        import torch
        rows = []
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                free_bytes, total_bytes = torch.cuda.mem_get_info(i)
                used_gb = (total_bytes - free_bytes) / (1024**3)
                total_gb = total_bytes / (1024**3)
                rows.append({
                    "gpu_index": i,
                    "name": torch.cuda.get_device_name(i),
                    "used_gb": round(used_gb, 2),
                    "free_gb": round(free_bytes / (1024**3), 2),
                    "total_gb": round(total_gb, 2),
                    "used_percent": round(used_gb / total_gb * 100, 2),
                })
        gpu_df = pd.DataFrame(rows)
        display(gpu_df)
    except Exception as exc:
        print("Torch GPU inspection failed:", exc)

    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True,
        )
        print(result.stdout)
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)


## 10) vLLM inference helpers

In [ ]:
def run_vllm_batch(sample: pd.DataFrame, show_prompt_preview: bool = True):
    llm = None
    rows = []
    try:
        llm = LLM(model=MODEL_NAME, **MODEL_KWARGS)
        html_box("GPU state after model load", "The table below shows the GPUs currently visible and their memory usage.")
        inspect_gpu_state()

        prompts = [build_prompt(v, llm=llm) for v in sample["model_input_text"].tolist()]
        if show_prompt_preview and prompts:
            print(prompts[0][:700])

        t0 = time.perf_counter()
        try:
            outs = llm.generate(prompts, sampling_params, use_tqdm=True)
        except TypeError:
            outs = llm.generate(prompts, sampling_params)
        t1 = time.perf_counter()

        elapsed = max(0.0, t1 - t0)
        rows_per_sec = len(sample) / elapsed if elapsed > 0 else float("inf")

        for in_row, out in zip(sample.itertuples(index=False), outs):
            raw_text = out.outputs[0].text if out.outputs else ""
            prediction, parse_ok, parse_error = parse_model_prediction(raw_text)
            rows.append({
                "row_id": int(getattr(in_row, "row_id")),
                "model_input_text": getattr(in_row, "model_input_text", ""),
                "model_name": MODEL_NAME,
                "raw_output": raw_text,
                "prediction": prediction,
                "parse_ok": parse_ok,
                "parse_error": parse_error,
            })
        return rows, elapsed, rows_per_sec
    finally:
        try:
            if llm is not None:
                del llm
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

def flatten_results(all_results: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    results_df = pd.DataFrame(all_results)
    scores_df = pd.DataFrame([
        {
            "row_id": item.get("row_id"),
            "model_input_text": item.get("model_input_text", ""),
            "informativeness": (item.get("prediction") or {}).get("informativeness"),
            "expressiveness": (item.get("prediction") or {}).get("expressiveness"),
            "phatic": (item.get("prediction") or {}).get("phatic"),
            "dominant_dimension": (item.get("prediction") or {}).get("dominant_dimension"),
            "confidence": (item.get("prediction") or {}).get("confidence"),
            "model_name": item.get("model_name", ""),
            "parse_ok": item.get("parse_ok", False),
        }
        for item in all_results
    ])
    return results_df, scores_df


## 11) Smoke test on 20 rows

In [ ]:
smoke_df = sample_df.head(SMOKE_TEST_N).copy()
html_box("Smoke test", f"Running a small smoke test on {len(smoke_df)} rows before the full corpus run.")

smoke_results, smoke_seconds, smoke_rps = run_vllm_batch(smoke_df, show_prompt_preview=True)
smoke_results_df, smoke_scores_df = flatten_results(smoke_results)

smoke_parse_ok = int(smoke_scores_df["parse_ok"].sum())
smoke_parse_rate = smoke_parse_ok / len(smoke_scores_df) * 100 if len(smoke_scores_df) else 0.0
html_box(
    "Smoke test summary",
    f"Rows: {len(smoke_scores_df)}\nRows/s: {smoke_rps:.2f}\nParse OK: {smoke_parse_ok}/{len(smoke_scores_df)} ({smoke_parse_rate:.1f}%)"
)
display(smoke_scores_df.head(10))


## 12) Full run

In [ ]:
html_box("Full run", f"Running the full corpus with {len(sample_df)} rows using {MODEL_NAME} on {TENSOR_PARALLEL_SIZE} GPUs.")
full_start_ts = time.perf_counter()
all_results, perf_total_seconds, perf_items_per_second = run_vllm_batch(sample_df, show_prompt_preview=False)
full_end_ts = time.perf_counter()

results_df, scores_df = flatten_results(all_results)

n = len(scores_df)
parse_ok_count = int(scores_df["parse_ok"].sum())
parse_fail_count = n - parse_ok_count
parse_ok_rate = parse_ok_count / n * 100 if n else 0.0

html_box(
    "Full run performance",
    f"Rows: {n}\nTotal seconds: {perf_total_seconds:.2f}\nRows per second: {perf_items_per_second:.2f}\nParse OK rate: {parse_ok_rate:.2f}%"
)


## 13) Convergence / stability check

In [ ]:
score_cols = ["informativeness", "expressiveness", "phatic"]
for col in score_cols:
    scores_df[col] = pd.to_numeric(scores_df[col], errors="coerce")

CHUNK_SIZE = 200
chunk_rows = []
for start in range(0, len(scores_df), CHUNK_SIZE):
    chunk = scores_df.iloc[start:start+CHUNK_SIZE].copy()
    if chunk.empty:
        continue
    chunk_rows.append({
        "chunk_id": len(chunk_rows) + 1,
        "n_rows": len(chunk),
        "parse_ok_rate": chunk["parse_ok"].mean() * 100,
        "informativeness_mean": chunk["informativeness"].mean(),
        "expressiveness_mean": chunk["expressiveness"].mean(),
        "phatic_mean": chunk["phatic"].mean(),
    })

chunk_summary_df = pd.DataFrame(chunk_rows)
display(chunk_summary_df.tail(10))

converged = False
convergence_reason = "Not enough chunks."
if len(chunk_summary_df) >= 3:
    last3 = chunk_summary_df.tail(3)
    deltas = {
        "informativeness": float(last3["informativeness_mean"].max() - last3["informativeness_mean"].min()),
        "expressiveness": float(last3["expressiveness_mean"].max() - last3["expressiveness_mean"].min()),
        "phatic": float(last3["phatic_mean"].max() - last3["phatic_mean"].min()),
    }
    parse_floor = float(last3["parse_ok_rate"].min())
    converged = max(deltas.values()) <= 0.15 and parse_floor >= 95.0
    convergence_reason = (
        f"Last 3 chunk deltas: info={deltas['informativeness']:.3f}, "
        f"expr={deltas['expressiveness']:.3f}, phatic={deltas['phatic']:.3f}; "
        f"min parse_ok_rate={parse_floor:.2f}%"
    )

if converged:
    html_box("Convergence status", "Model looks stable on the full run.\n" + convergence_reason, color="#2e7d32", bg="#edf7ed")
else:
    html_box("Convergence status", "Model does not clearly look converged yet.\n" + convergence_reason, color="#b26a00", bg="#fff7e6")


## 14) Save outputs

In [ ]:
jsonl_path = OUTPUT_ROOT / f"{MODEL_SLUG}__vllm_predictions_full_{n}.jsonl"
csv_full_path = OUTPUT_ROOT / f"{MODEL_SLUG}__vllm_predictions_full_{n}_full.csv"
csv_scores_path = OUTPUT_ROOT / f"{MODEL_SLUG}__vllm_predictions_full_{n}_scores.csv"

with open(jsonl_path, "w", encoding="utf-8") as f:
    for item in all_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

flat_df = results_df.copy()
flat_df["prediction"] = flat_df["prediction"].apply(lambda x: json.dumps(x, ensure_ascii=False))
flat_df.to_csv(csv_full_path, index=False)
scores_df.to_csv(csv_scores_path, index=False)

perf_row = {
    "model_name": MODEL_NAME,
    "model_slug": MODEL_SLUG,
    "column_strategy": COLUMN_STRATEGY,
    "rows": n,
    "rows_per_second": round(perf_items_per_second, 3),
    "parse_ok_rate_percent": round(parse_ok_rate, 2),
    "temperature": TEMPERATURE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_model_len": MAX_MODEL_LEN,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
    "converged": converged,
    "convergence_reason": convergence_reason,
    "scores_csv": str(csv_scores_path),
}
perf_df = pd.DataFrame([perf_row])
perf_path = OUTPUT_ROOT / f"{MODEL_SLUG}__run_summary.csv"
perf_df.to_csv(perf_path, index=False)

html_box(
    "Saved outputs",
    f"JSONL: {jsonl_path}\nFull CSV: {csv_full_path}\nScores CSV: {csv_scores_path}\nPerf summary: {perf_path}",
    color="#1b5e20",
    bg="#edf7ed",
)


## 15) Sanity check: how many rows were annotated?

In [ ]:
if "scores_df" not in globals() or scores_df.empty:
    scores_df = pd.read_csv(csv_scores_path)

annotated_rows = len(scores_df)
unique_row_ids = scores_df["row_id"].nunique()
missing_score_rows = int(scores_df[["informativeness", "expressiveness", "phatic"]].isna().any(axis=1).sum())
parse_ok_rows = int(scores_df["parse_ok"].fillna(False).sum())

html_box(
    "Sanity check",
    f"Rows annotated: {annotated_rows}\nUnique row_id: {unique_row_ids}\nRows with missing score values: {missing_score_rows}\nRows parsed successfully: {parse_ok_rows}"
)
display(scores_df.head(10))


## 16) Multi-model comparison setup

In [ ]:
# This cell scans existing score CSVs so you can compare multiple model runs together.
score_files = sorted(OUTPUT_ROOT.parent.rglob("*_scores.csv"))
comparison_frames = []
for path in score_files:
    try:
        tmp = pd.read_csv(path)
        if {"row_id", "informativeness", "expressiveness", "phatic", "model_name"}.issubset(tmp.columns):
            tmp["source_file"] = str(path)
            comparison_frames.append(tmp)
    except Exception:
        pass

if not comparison_frames:
    raise RuntimeError("No score CSVs found for model comparison.")

comparison_df = pd.concat(comparison_frames, ignore_index=True)
comparison_df["model_display"] = comparison_df["model_name"].fillna("unknown_model")
for col in ["informativeness", "expressiveness", "phatic"]:
    comparison_df[col] = pd.to_numeric(comparison_df[col], errors="coerce")

display(comparison_df[["model_display", "row_id", "informativeness", "expressiveness", "phatic"]].head(10))
print("Models found:", sorted(comparison_df["model_display"].dropna().unique().tolist()))


## 17) Graph 1: mean score comparison by model

In [ ]:
mean_by_model = comparison_df.groupby("model_display", as_index=False)[["informativeness", "expressiveness", "phatic"]].mean()
display(mean_by_model)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(mean_by_model))
width = 0.24
ax.bar(x - width, mean_by_model["informativeness"], width, label="informativeness", color="#C08081")
ax.bar(x, mean_by_model["expressiveness"], width, label="expressiveness", color="#5B8E7D")
ax.bar(x + width, mean_by_model["phatic"], width, label="phatic", color="#C7A64A")
ax.set_xticks(x)
ax.set_xticklabels(mean_by_model["model_display"], rotation=20, ha="right")
ax.set_ylim(1, 5)
ax.set_ylabel("Mean score")
ax.set_title("Mean score comparison by model")
ax.legend()
ax.grid(axis="y", alpha=0.25)
means_plot_path = LATEX_DIR / "means_by_model.png"
fig.savefig(means_plot_path, dpi=200, bbox_inches="tight")
print("Saved:", means_plot_path)
plt.tight_layout()
plt.show()


## 18) Graph 2: dominant-dimension shares by model

In [ ]:
dom_share = (
    comparison_df.groupby(["model_display", "dominant_dimension"])
    .size()
    .rename("n")
    .reset_index()
)
totals = dom_share.groupby("model_display")["n"].transform("sum")
dom_share["share_percent"] = dom_share["n"] / totals * 100
display(dom_share.head(20))

pivot_dom = dom_share.pivot(index="model_display", columns="dominant_dimension", values="share_percent").fillna(0)
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(pivot_dom))
colors = {"informativeness": "#C08081", "expressiveness": "#5B8E7D", "phatic": "#C7A64A", "mixed": "#7F7F7F"}
for col in ["informativeness", "expressiveness", "phatic", "mixed"]:
    if col in pivot_dom.columns:
        ax.bar(pivot_dom.index, pivot_dom[col], bottom=bottom, label=col, color=colors[col])
        bottom += pivot_dom[col].values
ax.set_ylabel("Share of rows (%)")
ax.set_title("Dominant-dimension shares by model")
ax.legend()
ax.grid(axis="y", alpha=0.25)
dom_plot_path = LATEX_DIR / "dominant_dimension_share_by_model.png"
fig.savefig(dom_plot_path, dpi=200, bbox_inches="tight")
print("Saved:", dom_plot_path)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 19) Graph 3: score distribution histograms by model

In [ ]:
models = sorted(comparison_df["model_display"].dropna().unique().tolist())
score_cols = ["informativeness", "expressiveness", "phatic"]
bins = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
palette = plt.cm.tab10(np.linspace(0, 1, max(3, len(models))))

for ax, col in zip(axes, score_cols):
    for model, color in zip(models, palette):
        vals = comparison_df.loc[comparison_df["model_display"] == model, col].dropna()
        ax.hist(vals, bins=bins, alpha=0.35, label=model, color=color)
    ax.set_title(col)
    ax.set_xticks([1,2,3,4,5])
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Count")
axes[-1].legend(fontsize=8)
dist_plot_path = LATEX_DIR / "score_distributions_by_model.png"
fig.savefig(dist_plot_path, dpi=200, bbox_inches="tight")
print("Saved:", dist_plot_path)
plt.tight_layout()
plt.show()


## 20) Graph 4: exact agreement and within-1-point agreement

In [ ]:
from itertools import combinations

rows = []
for model_a, model_b in combinations(sorted(comparison_df["model_display"].dropna().unique()), 2):
    left = comparison_df[comparison_df["model_display"] == model_a][["row_id", "informativeness", "expressiveness", "phatic"]].copy()
    right = comparison_df[comparison_df["model_display"] == model_b][["row_id", "informativeness", "expressiveness", "phatic"]].copy()
    merged = left.merge(right, on="row_id", suffixes=("_a", "_b"))
    if merged.empty:
        continue
    for col in ["informativeness", "expressiveness", "phatic"]:
        exact = (merged[f"{col}_a"] == merged[f"{col}_b"]).mean() * 100
        within1 = ((merged[f"{col}_a"] - merged[f"{col}_b"]).abs() <= 1).mean() * 100
        rows.append({
            "pair": f"{model_a} vs {model_b}",
            "dimension": col,
            "exact_agreement": exact,
            "within1_agreement": within1,
        })

agreement_df = pd.DataFrame(rows)
display(agreement_df.head(20))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, metric in zip(axes, ["exact_agreement", "within1_agreement"]):
    pivot = agreement_df.pivot(index="pair", columns="dimension", values=metric)
    x = np.arange(len(pivot))
    width = 0.24
    for idx, dim in enumerate(["informativeness", "expressiveness", "phatic"]):
        ax.bar(x + (idx-1)*width, pivot[dim], width, label=dim)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=25, ha="right")
    ax.set_title(metric.replace("_", " "))
    ax.set_ylabel("Percent")
    ax.grid(axis="y", alpha=0.25)
axes[1].legend()
agreement_plot_path = LATEX_DIR / "agreement_exact_within1_by_dimension.png"
fig.savefig(agreement_plot_path, dpi=200, bbox_inches="tight")
print("Saved:", agreement_plot_path)
plt.tight_layout()
plt.show()


## 21) Graph 5: expressiveness pair heatmap

In [ ]:
from itertools import combinations

pairs = []
for model_a, model_b in combinations(sorted(comparison_df["model_display"].dropna().unique()), 2):
    left = comparison_df[comparison_df["model_display"] == model_a][["row_id", "expressiveness"]].copy()
    right = comparison_df[comparison_df["model_display"] == model_b][["row_id", "expressiveness"]].copy()
    merged = left.merge(right, on="row_id", suffixes=("_a", "_b"))
    if merged.empty:
        continue
    exact = (merged["expressiveness_a"] == merged["expressiveness_b"]).mean()
    pairs.append((model_a, model_b, exact))

if not pairs:
    raise RuntimeError("Need at least two model outputs with overlapping row_id values.")

worst_pair = sorted(pairs, key=lambda x: x[2])[0]
model_a, model_b, _ = worst_pair
left = comparison_df[comparison_df["model_display"] == model_a][["row_id", "expressiveness"]].copy()
right = comparison_df[comparison_df["model_display"] == model_b][["row_id", "expressiveness"]].copy()
merged = left.merge(right, on="row_id", suffixes=("_a", "_b"))
heat = pd.crosstab(merged["expressiveness_a"], merged["expressiveness_b"]).reindex(index=[1,2,3,4,5], columns=[1,2,3,4,5], fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(heat.values, cmap="YlOrRd")
fig.colorbar(im, ax=ax)
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels([1,2,3,4,5])
ax.set_yticklabels([1,2,3,4,5])
ax.set_xlabel(model_b)
ax.set_ylabel(model_a)
ax.set_title("Expressiveness disagreement heatmap")
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(int(heat.values[i, j])), ha="center", va="center", color="black")
heatmap_path = LATEX_DIR / "expressiveness_pair_heatmap.png"
fig.savefig(heatmap_path, dpi=200, bbox_inches="tight")
print("Saved:", heatmap_path)
plt.tight_layout()
plt.show()


## 22) Graph 6: yearly trends by model

In [ ]:
year_source = df.loc[:, ["row_id", "year"]].copy()
year_source["year"] = pd.to_numeric(year_source["year"], errors="coerce")
trend_df = comparison_df.merge(year_source, on="row_id", how="left").dropna(subset=["year"]).copy()
trend_df["year"] = trend_df["year"].astype(int)

score_cols = ["informativeness", "expressiveness", "phatic"]
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for ax, col in zip(axes, score_cols):
    grouped = trend_df.groupby(["model_display", "year"], as_index=False)[col].mean()
    for model in sorted(grouped["model_display"].unique()):
        tmp = grouped[grouped["model_display"] == model]
        ax.plot(tmp["year"], tmp[col], marker="o", linewidth=1.8, markersize=4, label=model)
    ax.set_title(col)
    ax.set_xlabel("Year")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Mean score")
axes[-1].legend(fontsize=8)
yearly_plot_path = LATEX_DIR / "yearly_trends_by_model.png"
fig.savefig(yearly_plot_path, dpi=200, bbox_inches="tight")
print("Saved:", yearly_plot_path)
plt.tight_layout()
plt.show()
